In [0]:
CREATE OR REPLACE PROCEDURE dbw_healthcare_de_dev.etl.usp_manage_stage_audit
(
    IN p_action STRING,
    IN p_audit_id STRING,
    IN p_file_id STRING,
    IN p_adf_run_id STRING,
    IN p_stage STRING,
    IN p_source_count BIGINT,
    IN p_processed_count BIGINT,
    IN p_rejected_count BIGINT,
    IN p_error_message STRING
)
LANGUAGE SQL
SQL SECURITY INVOKER
AS
BEGIN

    IF UPPER(p_action) = 'START'
    THEN

        INSERT INTO dbw_healthcare_de_dev.etl.file_stage_audit
        (
            audit_id,
            file_id,
            batch_id,
            adf_run_id,
            processing_stage,
            attempt_number,
            stage_start_ts,
            stage_status,
            created_at
        )
        SELECT
            p_audit_id,
            fc.file_id,
            fc.batch_id,
            p_adf_run_id,
            UPPER(p_stage),

            COALESCE((
                SELECT MAX(a.attempt_number)
                FROM dbw_healthcare_de_dev.etl.file_stage_audit a
                WHERE a.file_id = p_file_id
                  AND a.processing_stage = UPPER(p_stage)
            ), 0) + 1,

            CURRENT_TIMESTAMP(),
            'IN_PROGRESS',
            CURRENT_TIMESTAMP()

        FROM dbw_healthcare_de_dev.etl.file_control fc

        WHERE fc.file_id = p_file_id
          AND fc.adf_run_id = p_adf_run_id
          AND fc.pipeline_status = 'IN_PROGRESS'
          AND fc.processing_stage = UPPER(p_stage)
          AND UPPER(p_stage) IN ('BRONZE', 'SILVER', 'GOLD')
          AND p_audit_id IS NOT NULL
          AND NOT EXISTS (
              SELECT 1
              FROM dbw_healthcare_de_dev.etl.file_stage_audit a
              WHERE a.audit_id = p_audit_id
                 OR (
                     a.file_id = p_file_id
                     AND a.adf_run_id = p_adf_run_id
                     AND a.processing_stage = UPPER(p_stage)
                     AND a.stage_status = 'IN_PROGRESS'
                 )
          );

    ELSEIF UPPER(p_action) IN ('SUCCESS', 'FAILED') THEN

        UPDATE dbw_healthcare_de_dev.etl.file_stage_audit
        SET
            stage_end_ts = CURRENT_TIMESTAMP(),

            duration_seconds =
                TIMESTAMPDIFF(
                    MILLISECOND,
                    stage_start_ts,
                    CURRENT_TIMESTAMP()
                ) / 1000.0,

            source_record_count = p_source_count,
            processed_record_count = p_processed_count,
            rejected_record_count = p_rejected_count,

            stage_status = UPPER(p_action),

            error_message =
                CASE
                    WHEN UPPER(p_action) = 'FAILED'
                    THEN p_error_message
                    ELSE NULL
                END

        WHERE audit_id = p_audit_id
          AND file_id = p_file_id
          AND adf_run_id = p_adf_run_id
          AND processing_stage = UPPER(p_stage)
          AND stage_status = 'IN_PROGRESS'
          AND stage_end_ts IS NULL;

    ELSE

        SIGNAL SQLSTATE '45000'
        SET MESSAGE_TEXT =
            'Invalid audit action. Use START, SUCCESS, or FAILED.';

    END IF;

END;